# Track B (SQuAD grounded QA) on Kaggle

Train and evaluate the corpus-grounded QA specialist on a free Kaggle T4 (16 GB),
where the long RAFT contexts are no longer cramped like on a 4 GB GPU.

## Setup (once)
1. Generate the data locally first (Phase 2 needs Ollama), then create a Kaggle
   Dataset from `track_b_squad_grounded/data/`: `seed.jsonl`, `train_synth.jsonl`,
   `gold.jsonl`. Name it **`track-b-data`** (mounts at `/kaggle/input/track-b-data/`).
2. This notebook: **Settings -> Accelerator -> GPU T4**, add the `track-b-data` dataset.
3. Run all cells.

If your dataset slug differs, edit `DATA` in the config cell.


In [ ]:
!pip -q install -U "transformers>=4.44" "trl>=0.9" peft bitsandbytes datasets accelerate
!pip -q uninstall -y torchao   # Kaggle ships an old torchao that recent peft rejects; we use bitsandbytes
print("deps installed")

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
DATA = "/kaggle/input/track-b-data"   # edit if your dataset slug differs
OUT  = "/kaggle/working"
print("DATA contents:", os.listdir(DATA) if os.path.isdir(DATA) else "NOT FOUND - add the dataset")

In [ ]:
import json, re, string

ABSTAIN = "not in the context"
SYSTEM_PROMPT = ("You are a question-answering assistant that uses ONLY the provided context. "
    "Read the documents and answer the question with the shortest exact phrase from the context "
    "that answers it. Do not use any outside knowledge. If the context does not contain the "
    "answer, reply with exactly: " + ABSTAIN)

def read_jsonl(p):
    with open(p) as f: return [json.loads(l) for l in f if l.strip()]
def user_of(r): return next(m["content"] for m in r["messages"] if m["role"] == "user")
def _na(s):
    s = s.lower(); s = "".join(c for c in s if c not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s); return " ".join(s.split())
def is_abstention(t):
    a = _na(ABSTAIN); n = _na(t); return n == a or n.startswith(a)
def exact_match(p, gs): return float(any(_na(p) == _na(g) for g in gs))
def token_f1(p, gs):
    def f1(p, g):
        pt, gt = _na(p).split(), _na(g).split()
        if not pt or not gt: return float(pt == gt)
        c = 0; pool = list(gt)
        for t in pt:
            if t in pool: c += 1; pool.remove(t)
        if not c: return 0.0
        pr, rc = c/len(pt), c/len(gt); return 2*pr*rc/(pr+rc)
    return max((f1(p, g) for g in gs), default=0.0)

SENTINEL = [{"question": "What is the capital of France?", "answers": ["paris"]}, {"question": "What is 7 times 8?", "answers": ["56"]}, {"question": "Who wrote the play Romeo and Juliet?", "answers": ["shakespeare"]}, {"question": "What is the chemical symbol for water?", "answers": ["h2o"]}, {"question": "How many continents are there on Earth?", "answers": ["7", "seven"]}, {"question": "What planet is known as the Red Planet?", "answers": ["mars"]}, {"question": "What is the largest ocean on Earth?", "answers": ["pacific"]}, {"question": "In what year did World War II end?", "answers": ["1945"]}, {"question": "What gas do plants absorb from the air during photosynthesis?", "answers": ["carbon dioxide", "co2"]}, {"question": "What is the square root of 144?", "answers": ["12"]}, {"question": "What language is mainly spoken in Brazil?", "answers": ["portuguese"]}, {"question": "How do you say 'hello' in Spanish?", "answers": ["hola"]}]
print("helpers ready,", len(SENTINEL), "sentinel probes")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # same base the local evaluator scores

def load_model(adapter=None):
    tok = AutoTokenizer.from_pretrained(BASE_MODEL); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
    if adapter:
        from peft import PeftModel; m = PeftModel.from_pretrained(m, adapter)
    return m.to("cuda").eval(), tok

@torch.no_grad()
def gen(model, tok, prompts, max_new, batch=32):
    outs = []
    for i in range(0, len(prompts), batch):
        ch = prompts[i:i+batch]
        txt = [tok.apply_chat_template(m, add_generation_prompt=True, tokenize=False) for m in ch]
        enc = tok(txt, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        g = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(ch)):
            outs.append(tok.decode(g[j][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
        print(f"  {min(i+batch,len(prompts))}/{len(prompts)}", end="\r")
    print()
    return outs
print("generation helpers ready")

## Train the two adapters (long RAFT contexts; max_len 1536)

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def train_one(data_file, name, max_len=1536, batch=1, grad_accum=16, epochs=3, lr=2e-4):
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant,
        torch_dtype=torch.bfloat16, device_map="auto")
    model.config.use_cache = False
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", target_modules="all-linear")
    ds = load_dataset("json", data_files=f"{DATA}/{data_file}", split="train")
    out = f"{OUT}/lora-{name}"
    cfg = SFTConfig(output_dir=out, num_train_epochs=epochs, per_device_train_batch_size=batch,
        gradient_accumulation_steps=grad_accum, learning_rate=lr, lr_scheduler_type="cosine",
        warmup_ratio=0.05, logging_steps=10, save_strategy="epoch", bf16=True,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=max_len, packing=False, assistant_only_loss=True, seed=0, report_to="none")
    tr = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora, processing_class=tok)
    r = tr.train(); tr.save_model(out); tok.save_pretrained(out)
    print(f"{name}: final_train_loss={r.training_loss:.4f} -> {out}")
    return out

train_one("seed.jsonl", "seed")              # control
train_one("train_synth.jsonl", "seed-synth") # real run

## Evaluate base vs seed vs seed-synth (EM/F1 + abstention + sentinel)

In [ ]:
def evaluate(name, adapter=None):
    gold = read_jsonl(f"{DATA}/gold.jsonl")
    m, tok = load_model(adapter)
    print(f"[{name}] grounded QA on {len(gold)} gold rows")
    prompts = [[x for x in r["messages"] if x["role"] != "assistant"] for r in gold]
    raw = gen(m, tok, prompts, 48)
    n_ans = n_un = 0; em = f1 = ans_ok = 0.0; abs_ok = hall = 0
    for r, o in zip(gold, raw):
        ab = is_abstention(o)
        if r["answerable"]:
            n_ans += 1; em += exact_match(o, r["answers"]); ff = token_f1(o, r["answers"])
            f1 += ff; ans_ok += (not ab) and ff >= 0.5
        else:
            n_un += 1; abs_ok += ab; hall += (not ab)
    n = len(gold); grounded = (ans_ok + abs_ok)/n
    sp = [[{"role":"user","content":x["question"]}] for x in SENTINEL]
    sraw = gen(m, tok, sp, 32)
    ok = sum(1 for x,o in zip(SENTINEL, sraw) if any(a in o.lower() for a in x["answers"]))
    res = {"name":name, "grounded_score":round(grounded,3),
           "em":round(em/n_ans,3) if n_ans else 0.0, "f1":round(f1/n_ans,3) if n_ans else 0.0,
           "abstention_accuracy":round(abs_ok/n_un,3) if n_un else 0.0,
           "hallucination_rate":round(hall/n_un,3) if n_un else 0.0,
           "sentinel":f"{ok}/{len(SENTINEL)}"}
    json.dump(res, open(f"{OUT}/result_{name}.json","w"), indent=2); print(" ", res)
    del m; import torch; torch.cuda.empty_cache()
    return res

R = [evaluate("base"),
     evaluate("seed", f"{OUT}/lora-seed"),
     evaluate("seed-synth", f"{OUT}/lora-seed-synth")]
print("\nname           grounded  EM     F1     abst_acc  halluc  sentinel")
for r in R:
    print(f"{r['name']:<14} {r['grounded_score']:.3f}     {r['em']:.3f}  {r['f1']:.3f}  "
          f"{r['abstention_accuracy']:.3f}     {r['hallucination_rate']:.3f}   {r['sentinel']}")

## Download the adapters + results

In [ ]:
import shutil
for n in ["seed", "seed-synth"]:
    shutil.make_archive(f"{OUT}/lora-{n}", "zip", f"{OUT}/lora-{n}")
print("Download lora-seed.zip / lora-seed-synth.zip and result_*.json from the Output tab.")
print("Unzip each into phase3_train/lora-<name>/ locally, then re-run eval there if you want.")